# 🚀 Delay Prediction Model Training
## Train on Google Colab with Google Drive Dataset

**Instructions:**
1. Upload this notebook to Google Colab
2. Runtime → Change runtime type → GPU (T4 recommended)
3. Run all cells
4. Download the trained model at the end

**Dataset:** 2.05GB CSV from Google Drive  
**File ID:** `1Yee5VXnbfNb8SBa0XOyUuMOz4rFnRR_s`

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q xgboost lightgbm catboost scikit-learn pandas numpy matplotlib seaborn joblib

## 🔗 Step 2: Mount Google Drive and Load Dataset

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive
print("📁 Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True)
print("✅ Drive mounted successfully!")

## 🔍 Step 3: Locate and Download Dataset

In [ ]:
# Method 1: Direct file access (if you know the path)
# Replace this path with the actual location in your Drive
# Example: '/content/drive/MyDrive/datasets/traffic_data.csv'

# Method 2: Use gdown to download by file ID
!pip install -q gdown
import gdown

file_id = '1Yee5VXnbfNb8SBa0XOyUuMOz4rFnRR_s'
output_path = '/content/dataset.csv'

print("⬇️ Downloading dataset from Google Drive...")
print("⏳ This may take 2-5 minutes for 2GB file...")

url = f'https://drive.google.com/uc?id={file_id}'
gdown.download(url, output_path, quiet=False)

print(f"✅ Dataset downloaded to: {output_path}")

## 📊 Step 4: Explore Dataset Structure

In [ ]:
print("🔬 Loading and exploring dataset...\n")
print("=" * 70)

# Load first few rows to inspect
df_sample = pd.read_csv(output_path, nrows=1000)

print("📋 DATASET INFO")
print("=" * 70)
print(f"Columns: {len(df_sample.columns)}")
print(f"Sample rows: {len(df_sample)}")
print(f"\n📝 Column Names:")
print(df_sample.columns.tolist())
print(f"\n📊 Data Types:")
print(df_sample.dtypes)
print(f"\n🔢 Sample Data (first 5 rows):")
print(df_sample.head())
print(f"\n📈 Statistical Summary:")
print(df_sample.describe())
print(f"\n❓ Missing Values:")
print(df_sample.isnull().sum())

# Try to identify target column
print(f"\n🎯 Identifying Target Variable...")
potential_targets = [col for col in df_sample.columns if any(keyword in col.lower() for keyword in ['delay', 'late', 'target', 'label', 'class'])]
if potential_targets:
    print(f"   Potential target columns: {potential_targets}")
else:
    print("   ⚠️ No obvious target column found. Will check all binary/categorical columns.")
    binary_cols = [col for col in df_sample.columns if df_sample[col].nunique() == 2]
    print(f"   Binary columns (could be target): {binary_cols}")

## ⚙️ Step 5: Configure Target Variable

**IMPORTANT:** After running the cell above, check the output and modify this cell to set the correct target column.

In [ ]:
# ===== CONFIGURE THIS =====
# Replace 'delay' with the actual target column name from the output above
TARGET_COLUMN = 'delay'  # Change this based on the dataset exploration output

# If target needs to be created (e.g., based on threshold), uncomment and modify:
# TARGET_COLUMN = 'custom_delay'
# df['custom_delay'] = (df['duration'] > df['expected_duration']).astype(int)

print(f"✅ Target column set to: {TARGET_COLUMN}")

## 🔄 Step 6: Load Full Dataset in Chunks & Preprocess

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

print("📥 Loading full dataset...")
print("⏳ This may take 5-10 minutes for 2GB CSV...\n")

# Load full dataset
df = pd.read_csv(output_path)

print(f"✅ Loaded dataset: {df.shape[0]:,} rows × {df.shape[1]} columns\n")
print("=" * 70)
print("🧹 PREPROCESSING")
print("=" * 70)

# Basic info
print(f"\n📊 Full Dataset Shape: {df.shape}")
print(f"💾 Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Check target distribution
if TARGET_COLUMN in df.columns:
    print(f"\n🎯 Target Distribution ({TARGET_COLUMN}):")
    print(df[TARGET_COLUMN].value_counts())
    print(f"   Percentage: {df[TARGET_COLUMN].value_counts(normalize=True) * 100}")
else:
    print(f"\n⚠️ WARNING: Target column '{TARGET_COLUMN}' not found!")
    print(f"Available columns: {df.columns.tolist()}")
    raise ValueError(f"Please set TARGET_COLUMN correctly in Step 5")

# Handle missing values
print(f"\n🔧 Handling missing values...")
missing_before = df.isnull().sum().sum()
df = df.dropna(subset=[TARGET_COLUMN])  # Drop rows with missing target

# Fill numeric missing values with median
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)

# Fill categorical missing values with mode
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df[col].isnull().any():
        df[col].fillna(df[col].mode()[0], inplace=True)

missing_after = df.isnull().sum().sum()
print(f"   Missing values: {missing_before:,} → {missing_after:,}")

# Encode categorical variables
print(f"\n🔤 Encoding categorical variables...")
label_encoders = {}
for col in categorical_cols:
    if col != TARGET_COLUMN:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le

print(f"   Encoded {len(categorical_cols)} categorical columns")

print(f"\n✅ Preprocessing complete!")
print(f"Final dataset shape: {df.shape}")

## ✂️ Step 7: Split Data

In [ ]:
print("✂️ Splitting data into train/test sets...\n")

# Separate features and target
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

# Save feature names
feature_names = X.columns.tolist()
print(f"📋 Features ({len(feature_names)}): {feature_names[:10]}...")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📊 Data Split:")
print(f"   Training: {X_train.shape[0]:,} samples")
print(f"   Testing:  {X_test.shape[0]:,} samples")
print(f"\n   Train class distribution:")
print(f"      {y_train.value_counts()}")
print(f"   Test class distribution:")
print(f"      {y_test.value_counts()}")

# Scale features
print(f"\n⚖️ Scaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Data ready for training!")

## 🤖 Step 8: Train Multiple Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import time

print("=" * 70)
print("🤖 MODEL TRAINING")
print("=" * 70)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=7, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=7, learning_rate=0.1, random_state=42, n_jobs=-1, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(n_estimators=200, max_depth=7, learning_rate=0.1, random_state=42, n_jobs=-1, verbose=-1)
}

results = []
trained_models = {}

for name, model in models.items():
    print(f"\n{'='*70}")
    print(f"🔄 Training {name}...")
    print(f"{'='*70}")
    
    start_time = time.time()
    
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else y_pred
    
    train_time = time.time() - start_time
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    try:
        auc = roc_auc_score(y_test, y_proba)
    except:
        auc = 0.5
    
    # Store results
    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
        'ROC-AUC': auc,
        'Train Time (s)': train_time
    })
    
    trained_models[name] = model
    
    # Print results
    print(f"\n✅ {name} Results:")
    print(f"   Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall:    {recall:.4f}")
    print(f"   F1 Score:  {f1:.4f}")
    print(f"   ROC-AUC:   {auc:.4f}")
    print(f"   Time:      {train_time:.2f}s")

# Create results dataframe
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('F1 Score', ascending=False)

print(f"\n{'='*70}")
print("📊 MODEL COMPARISON")
print(f"{'='*70}")
print(results_df.to_string(index=False))

best_model_name = results_df.iloc[0]['Model']
best_f1 = results_df.iloc[0]['F1 Score']
print(f"\n🏆 Best Model: {best_model_name} (F1: {best_f1:.4f})")

## 📊 Step 9: Detailed Evaluation of Best Model

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, confusion_matrix

print(f"\n{'='*70}")
print(f"📊 DETAILED EVALUATION: {best_model_name}")
print(f"{'='*70}")

best_model = trained_models[best_model_name]
y_pred_best = best_model.predict(X_test_scaled)
y_proba_best = best_model.predict_proba(X_test_scaled)[:, 1] if hasattr(best_model, 'predict_proba') else y_pred_best

# Classification Report
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred_best))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
print("\n🔢 Confusion Matrix:")
print(f"   TN={cm[0,0]:,}  FP={cm[0,1]:,}")
print(f"   FN={cm[1,0]:,}  TP={cm[1,1]:,}")

# Create visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title(f'Confusion Matrix - {best_model_name}', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba_best)
auc_score = roc_auc_score(y_test, y_proba_best)
axes[1].plot(fpr, tpr, linewidth=3, label=f'ROC Curve (AUC={auc_score:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random')
axes[1].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. Model Comparison Bar Chart
results_df_plot = results_df.set_index('Model')[['F1 Score', 'ROC-AUC', 'Accuracy']]
results_df_plot.plot(kind='bar', ax=axes[2], width=0.8)
axes[2].set_title('Model Comparison', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Score')
axes[2].set_ylim(0, 1)
axes[2].legend(loc='lower right')
axes[2].grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('/content/model_evaluation.png', dpi=300, bbox_inches='tight')
print("\n📊 Visualization saved: /content/model_evaluation.png")
plt.show()

# Feature Importance (if available)
if hasattr(best_model, 'feature_importances_'):
    print("\n🔥 Top 20 Most Important Features:")
    feature_importance = pd.DataFrame({
        'Feature': feature_names,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print(feature_importance.head(20).to_string(index=False))
    
    # Plot feature importance
    plt.figure(figsize=(10, 8))
    top_features = feature_importance.head(20)
    plt.barh(top_features['Feature'], top_features['Importance'])
    plt.xlabel('Importance')
    plt.title(f'Top 20 Feature Importances - {best_model_name}', fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('/content/feature_importance.png', dpi=300, bbox_inches='tight')
    print("\n📊 Feature importance saved: /content/feature_importance.png")
    plt.show()

## 💾 Step 10: Save Model and Artifacts

In [ ]:
import joblib
import json
from datetime import datetime

print("=" * 70)
print("💾 SAVING MODEL AND ARTIFACTS")
print("=" * 70)

# Save best model
model_filename = '/content/delay_predictor_best.pkl'
joblib.dump(best_model, model_filename)
print(f"\n✅ Best model saved: {model_filename}")

# Save scaler
scaler_filename = '/content/delay_predictor_scaler.pkl'
joblib.dump(scaler, scaler_filename)
print(f"✅ Scaler saved: {scaler_filename}")

# Save label encoders
encoders_filename = '/content/delay_predictor_encoders.pkl'
joblib.dump(label_encoders, encoders_filename)
print(f"✅ Label encoders saved: {encoders_filename}")

# Save configuration and results
config = {
    'model_name': best_model_name,
    'target_column': TARGET_COLUMN,
    'feature_names': feature_names,
    'num_features': len(feature_names),
    'training_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'dataset_size': df.shape[0],
    'train_samples': X_train.shape[0],
    'test_samples': X_test.shape[0],
    'metrics': {
        'accuracy': float(results_df.iloc[0]['Accuracy']),
        'precision': float(results_df.iloc[0]['Precision']),
        'recall': float(results_df.iloc[0]['Recall']),
        'f1_score': float(results_df.iloc[0]['F1 Score']),
        'roc_auc': float(results_df.iloc[0]['ROC-AUC'])
    },
    'all_models_comparison': results_df.to_dict('records')
}

config_filename = '/content/delay_predictor_config.json'
with open(config_filename, 'w') as f:
    json.dump(config, f, indent=2)
print(f"✅ Configuration saved: {config_filename}")

# Save results CSV
results_filename = '/content/model_comparison_results.csv'
results_df.to_csv(results_filename, index=False)
print(f"✅ Results CSV saved: {results_filename}")

print("\n" + "=" * 70)
print("✅ ALL FILES SAVED SUCCESSFULLY!")
print("=" * 70)
print("\n📦 Files to download:")
print("   1. delay_predictor_best.pkl (trained model)")
print("   2. delay_predictor_scaler.pkl (feature scaler)")
print("   3. delay_predictor_encoders.pkl (label encoders)")
print("   4. delay_predictor_config.json (configuration & metrics)")
print("   5. model_comparison_results.csv (all models comparison)")
print("   6. model_evaluation.png (visualizations)")
print("   7. feature_importance.png (if available)")
print("\n💡 Download these files and place them in your VS Code project's 'models/' folder")

## 📥 Step 11: Download Files to Your Computer

Run the cell below to zip all files for easy download:

In [ ]:
import zipfile
from google.colab import files

# Create zip file
zip_filename = '/content/delay_prediction_model.zip'
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    zipf.write('/content/delay_predictor_best.pkl', 'delay_predictor_best.pkl')
    zipf.write('/content/delay_predictor_scaler.pkl', 'delay_predictor_scaler.pkl')
    zipf.write('/content/delay_predictor_encoders.pkl', 'delay_predictor_encoders.pkl')
    zipf.write('/content/delay_predictor_config.json', 'delay_predictor_config.json')
    zipf.write('/content/model_comparison_results.csv', 'model_comparison_results.csv')
    zipf.write('/content/model_evaluation.png', 'model_evaluation.png')
    if Path('/content/feature_importance.png').exists():
        zipf.write('/content/feature_importance.png', 'feature_importance.png')

print("✅ All files zipped!")
print(f"📦 Zip file size: {Path(zip_filename).stat().st_size / 1024**2:.2f} MB")
print("\n⬇️ Downloading zip file...")

# Download zip file
files.download(zip_filename)

print("\n" + "=" * 70)
print("🎉 TRAINING COMPLETE!")
print("=" * 70)
print(f"\n🏆 Best Model: {best_model_name}")
print(f"📊 F1 Score: {best_f1:.4f}")
print(f"📊 ROC-AUC: {results_df.iloc[0]['ROC-AUC']:.4f}")
print(f"📊 Accuracy: {results_df.iloc[0]['Accuracy']:.4f}")
print("\n✅ Extract the zip file and place contents in your project's models/ folder")
print("✅ You can now use the model in your VS Code app!")

## 🧪 Step 12: Test the Model (Optional)

In [ ]:
# Test loading and prediction
print("🧪 Testing model loading and prediction...\n")

# Load model
loaded_model = joblib.load(model_filename)
loaded_scaler = joblib.load(scaler_filename)
loaded_encoders = joblib.load(encoders_filename)

# Test with a few samples
test_samples = X_test[:5]
test_samples_scaled = loaded_scaler.transform(test_samples)
predictions = loaded_model.predict(test_samples_scaled)
probabilities = loaded_model.predict_proba(test_samples_scaled)[:, 1] if hasattr(loaded_model, 'predict_proba') else predictions

print("Sample Predictions:")
for i, (pred, prob) in enumerate(zip(predictions, probabilities)):
    actual = y_test.iloc[i]
    print(f"  Sample {i+1}: Predicted={pred}, Probability={prob:.4f}, Actual={actual}, Match={'✅' if pred==actual else '❌'}")

print("\n✅ Model loading and prediction test successful!")